In [1]:
# =============================================================
# CELL 1 — Install dependencies & verify environment
# =============================================================
import subprocess, sys

def pip_install(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

# Install ultralytics if not present
try:
    import ultralytics
    print(f'ultralytics {ultralytics.__version__} already installed')
except ImportError:
    print('Installing ultralytics...')
    pip_install('ultralytics')
    import ultralytics
    print(f'ultralytics {ultralytics.__version__} installed')

# Install pydicom if not present
try:
    import pydicom
    print(f'pydicom {pydicom.__version__} already installed')
except ImportError:
    print('Installing pydicom...')
    pip_install('pydicom')
    import pydicom
    print(f'pydicom {pydicom.__version__} installed')

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU : {torch.cuda.get_device_name(0)}')
print('Environment check done.')

Installing ultralytics...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
ultralytics 8.4.21 installed
pydicom 3.0.1 already installed
PyTorch : 2.9.0+cu126
CUDA available : True
GPU : Tesla T4
Environment check done.


In [ ]:
# =============================================================
# CELL 2 — Imports, Paths, Global Config
# =============================================================
import os, gc, yaml, shutil
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut
import torch
import torch.nn as nn
from ultralytics import YOLO

# ── Paths ────────────────────────────────────────────────────
ROOT_DIR  = Path('data/raw/vinbigdata-chest-xray-abnormalities-detection')
TRAIN_DIR = ROOT_DIR / 'train'
TEST_DIR  = ROOT_DIR / 'test'
WORK_DIR  = Path('outputs')
YOLO_DIR  = WORK_DIR / 'yolo_dataset'
RUNS_DIR  = WORK_DIR / 'runs'
FOLD_DIR  = WORK_DIR / 'folds'
IMAGES_DIR = YOLO_DIR / 'images'
LABELS_DIR = YOLO_DIR / 'labels'

for d in [YOLO_DIR, RUNS_DIR, FOLD_DIR, IMAGES_DIR, LABELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Config ───────────────────────────────────────────────────
N_FOLDS   = 15    # ~1k val / ~14k train per fold
IMG_SIZE  = 640
EPOCHS    = 50
BATCH     = 16   # reduce to 8 if OOM
SEED      = 42
MODEL_NAME = 'yolo26x.pt'  # YOLO26 extra-large; NMS-free, MuSGD
DEVICE    = 0 if torch.cuda.is_available() else 'cpu'

# ── Class names (14 findings; 'No finding' = background) ────
CLASS_NAMES = [
    'Aortic enlargement', 'Atelectasis', 'Calcification',
    'Cardiomegaly', 'Consolidation', 'ILD', 'Infiltration',
    'Lung Opacity', 'Nodule/Mass', 'Other lesion',
    'Pleural effusion', 'Pleural thickening', 'Pneumothorax',
    'Pulmonary fibrosis',
]
CLASS2ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
NC = len(CLASS_NAMES)  # 14

# ── Load CSV ─────────────────────────────────────────────────
train_data = pd.read_csv(ROOT_DIR / 'train.csv')
print(f'Annotations     : {len(train_data):,} rows')
print(f'Unique images   : {train_data["image_id"].nunique():,}')
print(f'Columns         : {list(train_data.columns)}')
print(f'GPU available   : {torch.cuda.is_available()}')
print(f'Device          : {DEVICE}')
print(f'Classes         : {NC}')
print(train_data.head(3))

In [ ]:
# =============================================================
# CELL 3 — GPU DICOM Preprocessor + Custom Trainer
# Runs directly on GPU: DICOM pixels -> windowed RGB [0,1]
# =============================================================

class DicomPreprocessorGPU(nn.Module):
    """
    GPU-accelerated DICOM windowing + normalisation.
    Input : [B, 1, H, W] float32 (raw DICOM pixel values)
    Output: [B, 3, H, W] float32 in [0, 1]  (3-channel RGB copy)
    
    If learn_window=True the window center & width become
    nn.Parameters and are optimised jointly with the detector.
    """
    def __init__(self, learn_window: bool = False,
                 init_center: float = 0.45, init_width: float = 0.9):
        super().__init__()
        self.learn_window = learn_window
        wc = torch.tensor([init_center], dtype=torch.float32)
        ww = torch.tensor([init_width],  dtype=torch.float32)
        if learn_window:
            self.window_center = nn.Parameter(wc)
            self.window_width  = nn.Parameter(ww.clamp(min=0.05))
        else:
            self.register_buffer('window_center', wc)
            self.register_buffer('window_width',  ww)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape
        # 1. Per-image min-max normalise to [0,1]
        x_flat = x.view(B, -1)
        x_min  = x_flat.min(dim=1).values.view(B, 1, 1, 1)
        x_max  = x_flat.max(dim=1).values.view(B, 1, 1, 1)
        x_norm = (x - x_min) / (x_max - x_min + 1e-8)
        # 2. Soft VOI windowing via sigmoid
        wc = self.window_center.clamp(0.0, 1.0)
        ww = self.window_width.clamp(0.05, 2.0)
        k  = 4.0 / ww
        x_win = torch.sigmoid(k * (x_norm - wc))
        # 3. Re-stretch to [0,1]
        x_flat2 = x_win.view(B, -1)
        lo = x_flat2.min(dim=1).values.view(B, 1, 1, 1)
        hi = x_flat2.max(dim=1).values.view(B, 1, 1, 1)
        x_out = (x_win - lo) / (hi - lo + 1e-8)
        # 4. Grayscale -> 3-channel RGB
        return x_out.repeat(1, 3, 1, 1)  # [B,3,H,W]


# ── Subclass Ultralytics DetectionTrainer to apply GPU preprocessor ──
from ultralytics.models.yolo.detect import DetectionTrainer

class DicomDetectionTrainer(DetectionTrainer):
    """
    Drop-in replacement for YOLO DetectionTrainer that applies
    DicomPreprocessorGPU on each batch inside the GPU loop.
    Works with standard PNG images loaded by YOLO's dataloader;
    the preprocessor treats the already-loaded [B,3,H,W] images
    as if they were windowed from raw DICOM and enhances contrast.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.dicom_preproc = DicomPreprocessorGPU(
            learn_window=False, init_center=0.45, init_width=0.9)

    def preprocess_batch(self, batch):
        batch = super().preprocess_batch(batch)  # YOLO's standard step
        imgs  = batch['img']                      # [B, 3, H, W] float32 [0,1]
        # Convert to luminance, re-window, expand back to 3ch
        if imgs.mean() < 0.45:   # X-ray images tend to be dark
            gray    = (0.299*imgs[:,0:1] + 0.587*imgs[:,1:2] + 0.114*imgs[:,2:3])
            preproc = self.dicom_preproc.to(imgs.device)
            enhanced = preproc(gray)
            batch['img'] = enhanced
        return batch


# Quick sanity check
_dev = 'cuda' if torch.cuda.is_available() else 'cpu'
_preproc = DicomPreprocessorGPU(learn_window=False).to(_dev)
_dummy   = torch.randint(0, 4096, (2, 1, 512, 512), dtype=torch.float32).to(_dev)
_out     = _preproc(_dummy)
assert _out.shape == (2, 3, 512, 512), f'Unexpected shape: {_out.shape}'
assert 0.0 <= _out.min().item() and _out.max().item() <= 1.001
print(f'DicomPreprocessorGPU OK  |  device={_dev}')
print(f'Input : {tuple(_dummy.shape)}  range [{_dummy.min().item():.0f}, {_dummy.max().item():.0f}]')
print(f'Output: {tuple(_out.shape)}  range [{_out.min().item():.4f}, {_out.max().item():.4f}]')
print('DicomDetectionTrainer defined OK')
del _dummy, _out

In [ ]:
# =============================================================
# CELL 4 — DICOM → PNG conversion + YOLO label writing
# Idempotent: skips files already converted.
# Uses parallel CPU workers (ThreadPoolExecutor) to speed up.
# =============================================================
from concurrent.futures import ThreadPoolExecutor, as_completed

def dicom_to_png(dicom_path: Path, out_path: Path) -> tuple:
    """Convert one DICOM file to 8-bit PNG. Returns (width, height)."""
    try:
        ds   = pydicom.dcmread(str(dicom_path))
        data = apply_voi_lut(ds.pixel_array.astype(np.float32), ds)
    except Exception:
        data = ds.pixel_array.astype(np.float32)
    # Handle MONOCHROME1 (invert)
    if getattr(ds, 'PhotometricInterpretation', '') == 'MONOCHROME1':
        data = data.max() - data
    # Normalise to uint8
    lo, hi = data.min(), data.max()
    data = ((data - lo) / (hi - lo + 1e-8) * 255).astype(np.uint8)
    img = Image.fromarray(data).convert('RGB')
    img.save(str(out_path))
    return img.size  # (width, height)


def write_yolo_label(label_path: Path, anns: list):
    """Write YOLO .txt  (one line per box: class xc yc w h)"""
    with open(label_path, 'w') as f:
        for ann in anns:
            f.write(f'{ann[0]} {ann[1]:.6f} {ann[2]:.6f} {ann[3]:.6f} {ann[4]:.6f}\n')


# ── Build annotation lookup: image_id -> [(cls_id, xc, yc, bw, bh), ...] ──
all_image_ids = train_data['image_id'].unique()
findings = train_data[train_data['class_name'] != 'No finding'].copy()

# Build per-image dimension lookup from CSV
dims_lookup = (
    train_data.groupby('image_id')[['width','height']].first()
    if 'width' in train_data.columns and 'height' in train_data.columns
    else None
)

image_annotations = {img_id: [] for img_id in all_image_ids}

for img_id, grp in findings.groupby('image_id'):
    # Get image dimensions
    if dims_lookup is not None and img_id in dims_lookup.index:
        img_w = float(dims_lookup.loc[img_id, 'width'])
        img_h = float(dims_lookup.loc[img_id, 'height'])
    else:
        # Fallback: read directly from DICOM (slow but correct)
        ds    = pydicom.dcmread(str(TRAIN_DIR / f'{img_id}.dicom'))
        img_h, img_w = ds.pixel_array.shape[:2]
        img_w, img_h = float(img_w), float(img_h)

    anns = []
    for _, row in grp.iterrows():
        cls_id = CLASS2ID.get(row['class_name'])
        if cls_id is None:
            continue
        x1, y1, x2, y2 = float(row['x_min']), float(row['y_min']), float(row['x_max']), float(row['y_max'])
        xc = (x1 + x2) / 2.0 / img_w
        yc = (y1 + y2) / 2.0 / img_h
        bw = (x2 - x1) / img_w
        bh = (y2 - y1) / img_h
        xc, yc, bw, bh = [max(0.0, min(1.0, v)) for v in [xc, yc, bw, bh]]
        anns.append((cls_id, xc, yc, bw, bh))
    image_annotations[img_id] = anns

n_with = sum(1 for v in image_annotations.values() if len(v) > 0)
print(f'Images with findings : {n_with:,} / {len(all_image_ids):,}')
print(f'Images without       : {len(all_image_ids) - n_with:,} / {len(all_image_ids):,}')

# ── Convert DICOMs to PNG + write labels ──
already_done = len(list(IMAGES_DIR.glob('*.png')))
print(f'Already converted: {already_done} / {len(all_image_ids)}')

if already_done < len(all_image_ids):
    print('Starting DICOM -> PNG conversion (parallel, 4 workers)...')
    def convert_one(img_id):
        out_img = IMAGES_DIR / f'{img_id}.png'
        out_lbl = LABELS_DIR / f'{img_id}.txt'
        if not out_img.exists():
            dicom_to_png(TRAIN_DIR / f'{img_id}.dicom', out_img)
        write_yolo_label(out_lbl, image_annotations[img_id])
        return img_id

    with ThreadPoolExecutor(max_workers=4) as ex:
        futs = {ex.submit(convert_one, img_id): img_id for img_id in all_image_ids}
        for i, fut in enumerate(tqdm(as_completed(futs), total=len(all_image_ids), desc='Converting')):
            fut.result()  # raise exceptions if any
    print('Conversion complete!')
else:
    # Images exist; refresh label files in case annotations changed
    print('Images exist - refreshing labels...')
    for img_id in tqdm(all_image_ids, desc='Labels'):
        write_yolo_label(LABELS_DIR / f'{img_id}.txt', image_annotations[img_id])

print(f'Total PNGs  : {len(list(IMAGES_DIR.glob("*.png"))):,}')
print(f'Total labels: {len(list(LABELS_DIR.glob("*.txt"))):,}')

In [ ]:
# =============================================================
# CELL 5 — 15-Fold Cross-Validation Training with YOLO26
# Each fold: ~14k train / ~1k val
# Model sees ALL 15k images across all 15 folds.
# YOLO26: NMS-free end-to-end inference, MuSGD optimizer
# =============================================================

def write_fold_yaml(fold_num: int, train_ids: np.ndarray, val_ids: np.ndarray) -> str:
    """Write a YOLO dataset YAML for one fold, using symlinks."""
    fold_path = FOLD_DIR / f'fold_{fold_num:02d}'
    t_img = fold_path / 'train' / 'images'
    t_lbl = fold_path / 'train' / 'labels'
    v_img = fold_path / 'val'   / 'images'
    v_lbl = fold_path / 'val'   / 'labels'
    for d in [t_img, t_lbl, v_img, v_lbl]:
        d.mkdir(parents=True, exist_ok=True)

    def link_files(ids, img_dst, lbl_dst):
        for img_id in ids:
            src_img = IMAGES_DIR / f'{img_id}.png'
            src_lbl = LABELS_DIR / f'{img_id}.txt'
            dst_img = img_dst / f'{img_id}.png'
            dst_lbl = lbl_dst / f'{img_id}.txt'
            if not dst_img.exists() and src_img.exists():
                os.symlink(src_img, dst_img)
            if not dst_lbl.exists():
                if src_lbl.exists():
                    shutil.copy2(src_lbl, dst_lbl)
                else:
                    open(dst_lbl, 'w').close()  # empty = no findings

    link_files(train_ids, t_img, t_lbl)
    link_files(val_ids,   v_img, v_lbl)

    yaml_content = {
        'path'  : str(fold_path),
        'train' : 'train/images',
        'val'   : 'val/images',
        'nc'    : NC,
        'names' : CLASS_NAMES,
    }
    yaml_path = str(fold_path / 'dataset.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_content, f, default_flow_style=False)
    return yaml_path


# ── Stratified 15-fold split on image level ──
img_ids  = np.array(all_image_ids)
has_find = np.array([1 if len(image_annotations[i]) > 0 else 0 for i in img_ids])

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_splits = list(skf.split(img_ids, has_find))

print('Fold sizes:')
for fi, (tr, va) in enumerate(fold_splits):
    print(f'  Fold {fi+1:02d}: train={len(tr):,} | val={len(va):,}')

# ── Training loop ──
fold_results = []

for fold_idx, (train_indices, val_indices) in enumerate(fold_splits):
    fold_num   = fold_idx + 1
    train_ids  = img_ids[train_indices]
    val_ids    = img_ids[val_indices]

    print(f'\n{"="*60}')
    print(f' FOLD {fold_num:02d} / {N_FOLDS} | Train: {len(train_ids):,} | Val: {len(val_ids):,}')
    print(f'{"="*60}')

    yaml_path = write_fold_yaml(fold_num, train_ids, val_ids)

    # ─ YOLO26 training overrides ─
    # Key YOLO26 settings:
    #   optimizer='MuSGD'  — YOLO26's built-in Muon+SGD optimizer
    #   end2end=True       — NMS-free inference head (default for YOLO26)
    overrides = dict(
        model     = MODEL_NAME,
        data      = yaml_path,
        epochs    = EPOCHS,
        imgsz     = IMG_SIZE,
        batch     = BATCH,
        project   = str(RUNS_DIR),
        name      = f'fold_{fold_num:02d}',
        exist_ok  = True,
        pretrained= True,
        optimizer = 'MuSGD',      # YOLO26 preferred optimizer
        lr0       = 0.01,
        lrf       = 0.01,
        momentum  = 0.937,
        weight_decay = 0.0005,
        warmup_epochs = 3.0,
        box       = 7.5,
        cls       = 0.5,
        end2end   = True,         # NMS-free end-to-end YOLO26 head
        hsv_h     = 0.015,
        hsv_s     = 0.7,
        hsv_v     = 0.4,
        flipud    = 0.0,
        fliplr    = 0.5,
        mosaic    = 1.0,
        translate = 0.1,
        scale     = 0.5,
        close_mosaic = 10,
        amp       = True,
        device    = DEVICE,
        workers   = 4,
        seed      = SEED + fold_idx,
        verbose   = True,
        save      = True,
        save_period = 10,
        val       = True,
        plots     = True,
        cache     = False,
    )

    # Use DicomDetectionTrainer (GPU preprocessor) if on GPU, else plain YOLO
    if torch.cuda.is_available():
        trainer = DicomDetectionTrainer(overrides=overrides)
        trainer.train()
        results_dict = trainer.metrics
    else:
        model   = YOLO(MODEL_NAME)
        results = model.train(**{k: v for k, v in overrides.items() if k != 'model'})
        results_dict = results.results_dict if hasattr(results, 'results_dict') else {}

    map50    = results_dict.get('metrics/mAP50(B)',    results_dict.get('mAP50',    None))
    map5095  = results_dict.get('metrics/mAP50-95(B)', results_dict.get('mAP50-95', None))

    fold_results.append({
        'fold'      : fold_num,
        'map50'     : map50,
        'map50_95'  : map5095,
        'best_model': str(RUNS_DIR / f'fold_{fold_num:02d}' / 'weights' / 'best.pt'),
    })
    print(f'Fold {fold_num:02d} -> mAP50={map50}  mAP50-95={map5095}')

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ── Summary ──
print('\n' + '='*60)
print(' CROSS-VALIDATION SUMMARY')
print('='*60)
results_df = pd.DataFrame(fold_results)
print(results_df.to_string(index=False))
print(f"\n  Mean mAP@50    : {results_df['map50'].mean():.4f}")
print(f"  Mean mAP@50-95 : {results_df['map50_95'].mean():.4f}")
print('='*60)
results_df.to_csv(WORK_DIR / 'cv_results.csv', index=False)
print(f'CV results saved -> {WORK_DIR}/cv_results.csv')

In [ ]:
# =============================================================
# CELL 6 — Inference on test set + submission CSV
# Uses the best fold model (highest mAP50).
# =============================================================

# ── Pick best fold ──
try:
    results_df
except NameError:
    results_df = pd.read_csv(WORK_DIR / 'cv_results.csv')

best_row   = results_df.loc[results_df['map50'].idxmax()]
best_fold  = int(best_row['fold'])
best_model_path = str(RUNS_DIR / f'fold_{best_fold:02d}' / 'weights' / 'best.pt')
print(f'Best fold : {best_fold:02d} | mAP50 = {best_row["map50"]:.4f}')
print(f'Model     : {best_model_path}')

# ── Convert test DICOMs to PNG (same pipeline) ──
TEST_IMAGES_DIR = YOLO_DIR / 'test_images'
TEST_IMAGES_DIR.mkdir(exist_ok=True)

test_dcms = list(TEST_DIR.glob('*.dicom'))
print(f'Test DICOMs: {len(test_dcms)}')

for dcm in tqdm(test_dcms, desc='Test conversion'):
    out = TEST_IMAGES_DIR / f'{dcm.stem}.png'
    if not out.exists():
        dicom_to_png(dcm, out)

print(f'Test PNGs: {len(list(TEST_IMAGES_DIR.glob("*.png")))}')

# ── Load best model & predict ──
best_model = YOLO(best_model_path)

predictions = best_model.predict(
    source   = str(TEST_IMAGES_DIR),
    imgsz    = IMG_SIZE,
    conf     = 0.25,
    iou      = 0.45,
    save     = False,
    save_txt = False,
    device   = DEVICE,
    verbose  = False,
)

# ── Build submission CSV ──
# VinBigData format: image_id, PredictionString
# PredictionString per finding: "class_id confidence x_min y_min x_max y_max"
# For no-finding images: "14 1.0 0 0 1 1"
sub_rows = []
for result in predictions:
    img_id = Path(result.path).stem
    boxes  = result.boxes
    if boxes is None or len(boxes) == 0:
        sub_rows.append({'image_id': img_id, 'PredictionString': '14 1.0 0 0 1 1'})
        continue
    preds = []
    for box in boxes:
        cls_id = int(box.cls.item())
        conf   = float(box.conf.item())
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        preds.append(f'{cls_id} {conf:.4f} {int(x1)} {int(y1)} {int(x2)} {int(y2)}')
    sub_rows.append({'image_id': img_id, 'PredictionString': ' '.join(preds)})

submission = pd.DataFrame(sub_rows)
out_path   = WORK_DIR / 'submission.csv'
submission.to_csv(out_path, index=False)
print(f'Submission saved: {len(submission):,} rows -> {out_path}')
print(submission.head(5))